# Generative RAG Baseline — Unified Batch Run and RQ2 Comparison

This notebook runs the **generative baseline answerer** on both **Rev5** and **Rev4**, saves baseline batch artifacts, and compares them against the latest **ComplianceGPT** batch outputs.

Design goal:
- keep retrieval fixed
- change only final answer construction
- report committee-friendly failure metrics that make baseline weaknesses explicit


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import os, sys, json, time, glob
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC_DIR =", SRC_DIR)


PROJECT_ROOT = /content/drive/MyDrive/ComplianceGPT_v2
SRC_DIR = /content/drive/MyDrive/ComplianceGPT_v2/src


In [3]:

# --- Canonical paths (Drive) ---
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

QUR_REV5_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev5.csv")
QUR_REV4_PATH = os.path.join(DATA_DIR, "qur_outputs", "qur_rewrites_rev4.csv")

BASELINE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "experiments", "answerer_comparison", "baseline_runs")
COMPARE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "experiments", "answerer_comparison", "comparison_reports")
COMPLIANCE_RUN_DIR = os.path.join(PROJECT_ROOT, "experiments", "pipeline_runs")
COMPLIANCE_REPORT_DIR = os.path.join(COMPLIANCE_RUN_DIR, "reports")

for p in [REV5_CATALOG_PATH, REV4_CATALOG_PATH, REV5_GOLD_PATH, REV4_GOLD_PATH]:
    print("[OK]" if os.path.exists(p) else "[WARN]", p)


[OK] /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
[OK] /content/drive/MyDrive/ComplianceGPT_v2/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
[OK] /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv
[OK] /content/drive/MyDrive/ComplianceGPT_v2/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv


In [4]:

from typing import Any

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    df = pd.read_csv(path, encoding="latin1")
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def load_qur_map(framework_version: str) -> dict[str, list[str]]:
    path = QUR_REV5_PATH if framework_version.lower() == "rev5" else QUR_REV4_PATH
    if not os.path.exists(path):
        print(f"[INFO] QUR CSV not found: {path}")
        return {}
    df = pd.read_csv(path)
    required = {"query_id", "rewrite_rank", "rewritten_query"}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"Unexpected QUR CSV schema. Missing {required - set(df.columns)}. Columns={list(df.columns)}")
    out: dict[str, list[str]] = {}
    for qid, g in df.groupby(df["query_id"].astype(str)):
        g2 = g.sort_values("rewrite_rank")
        out[str(qid)] = [str(x) for x in g2["rewritten_query"].tolist() if isinstance(x, str) and x.strip()]
    return out

def _safe_json_load(x):
    if not isinstance(x, str) or not x.strip():
        return {}
    try:
        return json.loads(x)
    except Exception:
        return {}

def _count_gold_clause_ids(x) -> int:
    if x is None:
        return 0
    if isinstance(x, float) and np.isnan(x):
        return 0
    if isinstance(x, list):
        return len([v for v in x if str(v).strip()])
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return 0
    if s.startswith("[") and s.endswith("]"):
        try:
            arr = json.loads(s)
            if isinstance(arr, list):
                return len([v for v in arr if str(v).strip()])
        except Exception:
            pass
    if "|" in s:
        return len([t for t in s.split("|") if t.strip()])
    return len([t for t in s.splitlines() if t.strip()])

def _to_float(x):
    try:
        return float(x)
    except Exception:
        return float("nan")

def evaluate_contract_df(df_out: pd.DataFrame) -> pd.DataFrame:
    contracts = df_out["contract_json"].apply(_safe_json_load)
    verifier_metrics = contracts.apply(lambda c: c.get("verifier_metrics") or {})
    vm_df = pd.json_normalize(verifier_metrics).add_prefix("vm.")

    df_eval = df_out.copy()
    df_eval["evidence_span_count"] = contracts.apply(lambda c: len(c.get("evidence_spans") or []))
    df_eval["has_evidence_spans"] = df_eval["evidence_span_count"] > 0
    df_eval = pd.concat([df_eval, vm_df], axis=1)

    df_eval["gold_clause_count"] = df_eval.get("gold_control_path", "").apply(_count_gold_clause_ids) if "gold_control_path" in df_eval.columns else 0
    df_eval["control_recall"] = df_eval.get("vm.control_recall", np.nan).apply(_to_float) if "vm.control_recall" in df_eval.columns else float("nan")
    df_eval["doc_recall"] = df_eval.get("vm.doc_recall", np.nan).apply(_to_float) if "vm.doc_recall" in df_eval.columns else float("nan")
    df_eval["verbatim_strict_pass_rate"] = df_eval.get("vm.verbatim_strict_pass_rate", np.nan).apply(_to_float) if "vm.verbatim_strict_pass_rate" in df_eval.columns else float("nan")
    df_eval["verbatim_normalized_pass_rate"] = df_eval.get("vm.verbatim_normalized_pass_rate", np.nan).apply(_to_float) if "vm.verbatim_normalized_pass_rate" in df_eval.columns else float("nan")

    df_eval["control_hit_any"] = df_eval["control_recall"].fillna(0.0) > 0.0
    df_eval["control_full_recall"] = df_eval["control_recall"].fillna(0.0) >= (1.0 - 1e-9)
    df_eval["doc_hit_any"] = df_eval["doc_recall"].fillna(0.0) > 0.0
    df_eval["doc_full_recall"] = df_eval["doc_recall"].fillna(0.0) >= (1.0 - 1e-9)
    return df_eval

def find_latest(pattern: str) -> str:
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")
    files = sorted(files, key=os.path.getmtime, reverse=True)
    return files[0]

print("[OK] Helper functions loaded.")


[OK] Helper functions loaded.


In [5]:

from generative_answerer.pipeline import BaselineGenerativeRAGPipeline
from dataclasses import replace

def build_baseline_pipeline(
    *,
    framework_version: str,
    use_qur_live: bool,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
):
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
    pipe = BaselineGenerativeRAGPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=bool(use_qur_live),
        org_profile_path=None,
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


## Configure unified batch run

In [6]:

# === USER CONFIG ===
FRAMEWORKS = ["rev5", "rev4"]
USE_QUR_PRECOMPUTED = True
USE_QUR_LIVE = False
USE_ORG_PROFILE = False

VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12
MAX_ROWS_BY_FRAMEWORK = {
    "rev5": None,
    "rev4": None,
}

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
os.makedirs(BASELINE_OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPARE_OUTPUT_DIR, exist_ok=True)

print("RUN_TAG =", RUN_TAG)
print("FRAMEWORKS =", FRAMEWORKS)


RUN_TAG = 20260313_121821
FRAMEWORKS = ['rev5', 'rev4']


In [7]:

from tqdm.auto import tqdm

def _gold_field(gold_row: dict, keys: list[str], default=""):
    for k in keys:
        if k in gold_row and gold_row.get(k) not in (None, ""):
            return gold_row.get(k)
    return default

def _stringify_maybe_list(x) -> str:
    if x is None:
        return ""
    if isinstance(x, list):
        return "|".join([str(v) for v in x])
    return str(x)

def run_baseline_for_framework(framework_version: str) -> dict:
    gold_df = load_gold(framework_version)
    max_rows = MAX_ROWS_BY_FRAMEWORK.get(framework_version)
    if max_rows is not None:
        gold_df = gold_df.head(int(max_rows))

    qur_map = load_qur_map(framework_version) if USE_QUR_PRECOMPUTED else {}
    pipe = build_baseline_pipeline(
        framework_version=framework_version,
        use_qur_live=USE_QUR_LIVE,
        use_org_profile=USE_ORG_PROFILE,
        verify_strict_extras=VERIFY_STRICT_EXTRAS,
        verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
        verify_strict_version=VERIFY_STRICT_VERSION,
    )

    if hasattr(pipe, "retriever") and hasattr(pipe.retriever, "cfg"):
        try:
            pipe.retriever.cfg = replace(pipe.retriever.cfg, top_k=int(TOP_K))
        except Exception:
            pass

    rows = []
    for _, r in tqdm(gold_df.iterrows(), total=len(gold_df), desc=f"Baseline batch ({framework_version})"):
        gold_row = dict(r.to_dict())
        qid = str(gold_row.get("query_id", gold_row.get("id", "")))
        query = str(gold_row.get("original_query", gold_row.get("question", ""))).strip()
        rewrites = qur_map.get(qid, []) if USE_QUR_PRECOMPUTED else []

        out = pipe.answer(
            query,
            top_k=int(TOP_K),
            rewrites=rewrites if rewrites else None,
            gold_row=gold_row,
            use_generator=True,
            run_verify=True,
        )
        c = out.get("contract", out)

        rows.append({
            "query_id": qid,
            "framework_version": framework_version,
            "question": query,
            "gold_control_id": _stringify_maybe_list(_gold_field(gold_row, ["control_id", "gold_control_id", "control"], default="")),
            "gold_source_version": _stringify_maybe_list(_gold_field(gold_row, ["source_version", "gold_source_version", "framework_version"], default="")),
            "gold_resolution_policy": _stringify_maybe_list(_gold_field(gold_row, ["resolution_policy", "gold_resolution_policy"], default="")),
            "gold_control_path": _stringify_maybe_list(_gold_field(gold_row, ["gold_control_path", "control_path", "doc_ids", "doc_id_list", "gold_doc_ids"], default="")),
            "gold_odp_required": _stringify_maybe_list(_gold_field(gold_row, ["odp_required"], default="")),
            "status": c.get("status"),
            "verifier_pass": bool(c.get("verifier_pass")),
            "verifier_errors": "|".join(list(c.get("verifier_errors") or [])),
            "primary_citation": c.get("primary_citation", ""),
            "all_citations": c.get("all_citations", ""),
            "odp_required_list": "|".join(list(c.get("odp_required_list") or [])),
            "selected_source_ids": "|".join(list(c.get("selected_source_ids") or [])),
            "contract_json": json.dumps(c, ensure_ascii=False),
        })

    df_out = pd.DataFrame(rows)
    df_eval = evaluate_contract_df(df_out)

    contracts_path = os.path.join(BASELINE_OUTPUT_DIR, f"generative_baseline_{framework_version}_contracts_{RUN_TAG}.csv")
    eval_path = os.path.join(BASELINE_OUTPUT_DIR, f"generative_baseline_{framework_version}_eval_{RUN_TAG}.csv")
    df_out.to_csv(contracts_path, index=False)
    df_eval.to_csv(eval_path, index=False)

    print("[OK]", framework_version, "contracts ->", contracts_path)
    print("[OK]", framework_version, "eval      ->", eval_path)
    return {
        "framework_version": framework_version,
        "gold_df": gold_df,
        "contracts_df": df_out,
        "eval_df": df_eval,
        "contracts_path": contracts_path,
        "eval_path": eval_path,
    }

baseline_runs = {}
for fw in FRAMEWORKS:
    baseline_runs[fw] = run_baseline_for_framework(fw)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Pipeline] 4-bit requested but bitsandbytes unavailable. Falling back to full precision. (No module named 'bitsandbytes')


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Baseline batch (rev4):   0%|          | 0/36 [00:00<?, ?it/s]

[OK] rev4 contracts -> /content/drive/MyDrive/ComplianceGPT_v2/experiments/answerer_comparison/baseline_runs/generative_baseline_rev4_contracts_20260313_121821.csv
[OK] rev4 eval      -> /content/drive/MyDrive/ComplianceGPT_v2/experiments/answerer_comparison/baseline_runs/generative_baseline_rev4_eval_20260313_121821.csv


## Compare baseline vs latest ComplianceGPT run

In [8]:

from answerer_comparison.answerer_compare import compare_answerers

comparison_outputs = {}

for fw in FRAMEWORKS:
    baseline_contracts = baseline_runs[fw]["contracts_df"]
    baseline_eval = baseline_runs[fw]["eval_df"]
    gold_df = baseline_runs[fw]["gold_df"]

    compliance_contracts_path = find_latest(os.path.join(COMPLIANCE_RUN_DIR, f"pipeline_{fw}_contracts_*.csv"))
    compliance_eval_path = find_latest(os.path.join(COMPLIANCE_REPORT_DIR, f"eval_{fw}_*.csv"))

    compliance_contracts = pd.read_csv(compliance_contracts_path)
    compliance_eval = pd.read_csv(compliance_eval_path)

    out_dir = os.path.join(COMPARE_OUTPUT_DIR, fw)
    comparison_outputs[fw] = compare_answerers(
        baseline_eval=baseline_eval,
        baseline_contracts=baseline_contracts,
        compliance_eval=compliance_eval,
        compliance_contracts=compliance_contracts,
        gold_df=gold_df,
        framework_version=fw,
        output_dir=out_dir,
        ts=RUN_TAG,
    )

    print("\n==============================")
    print("Framework:", fw)
    print("ComplianceGPT contracts:", compliance_contracts_path)
    print("ComplianceGPT eval     :", compliance_eval_path)
    print("==============================")
    display(comparison_outputs[fw]["system_table"])
    display(comparison_outputs[fw]["head_to_head"])
    display(comparison_outputs[fw]["status_matrix"].head(20))
    display(comparison_outputs[fw]["failure_examples"].head(10))



Framework: rev5
ComplianceGPT contracts: /content/drive/MyDrive/ComplianceGPT_v2/experiments/pipeline_runs/pipeline_rev5_contracts_20260312_150620.csv
ComplianceGPT eval     : /content/drive/MyDrive/ComplianceGPT_v2/experiments/pipeline_runs/reports/eval_rev5_20260312_150620.csv


,system,n_questions,audit_ready_answer_rate,right_control_found_rate,right_clause_found_rate,full_gold_clause_coverage_rate,answer_has_supporting_evidence_rate,average_supporting_span_count,strict_verbatim_grounding_rate,normalized_verbatim_grounding_rate,n_odp_questions,odp_correct_handling_rate,odp_false_complete_rate,odp_unresolved_detected_rate,odp_unresolved_hidden_rate
0,ComplianceGPT,100,0.64,0.96,0.92,0.64,1.00,4.01,1.0,1.0,63,1.0,0.000000,1.000000,0.000000
1,Generative RAG baseline,100,0.22,0.91,0.77,0.48,0.95,3.39,1.0,1.0,63,0.0,0.952381,0.365079,0.634921


,n_compared_questions,compliancegpt_strict_win_rate,generative_false_complete_vs_compliancegpt_safe_warning_rate,generative_missed_right_control_vs_compliancegpt_rate,same_final_status_rate
0,100,0.44,0.78,0.05,0.17


,baseline_status,compliancegpt_status,n
0,OK,OK,17
1,OK,PARAMS_REQUIRED,78
2,NO_EVIDENCE,PARAMS_REQUIRED,2
3,ERROR,PARAMS_REQUIRED,3


,query_id,framework_version,question,gold_has_odp,baseline_status,compliancegpt_status,baseline_verifier_pass,compliancegpt_verifier_pass,baseline_primary_citation,compliancegpt_primary_citation,baseline_answer_preview,compliancegpt_answer_preview
0,1,rev5,When must account managers and the designated ...,True,OK,PARAMS_REQUIRED,False,True,ac-2_smt.h.1,ac-2_smt.h,,
1,100,rev5,What policies and procedures should be develop...,True,OK,PARAMS_REQUIRED,False,True,sr-11_smt,sr-11_smt,,
2,11,rev5,What are the organizational responsibilities f...,True,OK,PARAMS_REQUIRED,False,True,au-2_smt,au-2_smt.d,,
3,14,rev5,What is the required process for reviewing sys...,True,OK,PARAMS_REQUIRED,False,True,au-6_smt,au-6_gdn,,
4,15,rev5,What are the requirements for generating and r...,True,OK,PARAMS_REQUIRED,False,True,au-8_smt,au-8_smt.a,,
5,16,rev5,Describe the key steps of the formal control a...,True,OK,PARAMS_REQUIRED,False,False,ca-2_smt.a,ca-2_smt.f,,
6,17,rev5,What are the requirements for managing and sec...,True,OK,PARAMS_REQUIRED,False,True,ca-3_smt,ca-3_gdn,,
7,18,rev5,What is the purpose of a plan of action and mi...,True,OK,PARAMS_REQUIRED,False,True,ca-5_smt.a,ca-5_smt,,
8,19,rev5,What are the primary responsibilities of an au...,True,OK,PARAMS_REQUIRED,False,False,ca-6_gdn,ca-6_gdn,,
9,20,rev5,What are the requirements for implementing a s...,True,OK,PARAMS_REQUIRED,False,True,ca-7_smt,ca-7_smt,,



Framework: rev4
ComplianceGPT contracts: /content/drive/MyDrive/ComplianceGPT_v2/experiments/pipeline_runs/pipeline_rev4_contracts_20260312_152654.csv
ComplianceGPT eval     : /content/drive/MyDrive/ComplianceGPT_v2/experiments/pipeline_runs/reports/eval_rev4_20260312_152654.csv


,system,n_questions,audit_ready_answer_rate,right_control_found_rate,right_clause_found_rate,full_gold_clause_coverage_rate,answer_has_supporting_evidence_rate,average_supporting_span_count,strict_verbatim_grounding_rate,normalized_verbatim_grounding_rate,n_odp_questions,odp_correct_handling_rate,odp_false_complete_rate,odp_unresolved_detected_rate,odp_unresolved_hidden_rate
0,ComplianceGPT,36,0.750000,1.0,0.972222,0.750000,1.0,4.222222,1.0,1.0,19,1.0,0.0,1.000000,0.000000
1,Generative RAG baseline,36,0.222222,1.0,0.833333,0.555556,1.0,2.722222,1.0,1.0,19,0.0,1.0,0.315789,0.684211


,n_compared_questions,compliancegpt_strict_win_rate,generative_false_complete_vs_compliancegpt_safe_warning_rate,generative_missed_right_control_vs_compliancegpt_rate,same_final_status_rate
0,36,0.527778,0.777778,0.0,0.222222


,baseline_status,compliancegpt_status,n
0,OK,OK,8
1,OK,PARAMS_REQUIRED,28


,query_id,framework_version,question,gold_has_odp,baseline_status,compliancegpt_status,baseline_verifier_pass,compliancegpt_verifier_pass,baseline_primary_citation,compliancegpt_primary_citation,baseline_answer_preview,compliancegpt_answer_preview
0,1,rev4,What does AC-2 require the organization to do ...,True,OK,PARAMS_REQUIRED,False,True,ac-2_smt,ac-2_smt.a,,
1,10,rev4,What are the requirements for establishing and...,True,OK,PARAMS_REQUIRED,False,True,cm-6_smt.a,cm-6_smt.a,,
2,12,rev4,What does CP-9 require for conducting and prot...,True,OK,PARAMS_REQUIRED,False,True,cp-9_smt,cp-9_smt,,
3,16,rev4,What does IR-6 require about reporting suspect...,True,OK,PARAMS_REQUIRED,False,False,ir-6_smt,ir-6_smt.a,,
4,18,rev4,Who must approve removing the system or compon...,True,OK,PARAMS_REQUIRED,False,True,ma-2_smt.c,ma-2_smt.c,,
5,19,rev4,What does MP-4 require for protecting informat...,True,OK,PARAMS_REQUIRED,False,True,mp-4_smt,mp-4_smt.b,,
6,20,rev4,What does MP-6 require for sanitizing media pr...,True,OK,PARAMS_REQUIRED,False,True,mp-6_smt,mp-6_smt,,
7,21,rev4,What are the requirements for enforcing physic...,True,OK,PARAMS_REQUIRED,False,False,pe-3_smt.a.1,pe-3_smt.c,,
8,28,rev4,What does PS-3 require before authorizing acce...,True,OK,PARAMS_REQUIRED,False,True,ps-3_smt.a,ps-3_smt,,
9,3,rev4,What security awareness training must the orga...,True,OK,PARAMS_REQUIRED,False,True,at-2_smt,at-2_smt,,


In [9]:

# Combined cross-framework view
combined_system = []
combined_head = []

for fw in FRAMEWORKS:
    st = comparison_outputs[fw]["system_table"].copy()
    st.insert(0, "framework_version", fw)
    combined_system.append(st)

    hh = comparison_outputs[fw]["head_to_head"].copy()
    hh.insert(0, "framework_version", fw)
    combined_head.append(hh)

combined_system_df = pd.concat(combined_system, ignore_index=True) if combined_system else pd.DataFrame()
combined_head_df = pd.concat(combined_head, ignore_index=True) if combined_head else pd.DataFrame()

display(combined_system_df)
display(combined_head_df)


,framework_version,system,n_questions,audit_ready_answer_rate,right_control_found_rate,right_clause_found_rate,full_gold_clause_coverage_rate,answer_has_supporting_evidence_rate,average_supporting_span_count,strict_verbatim_grounding_rate,normalized_verbatim_grounding_rate,n_odp_questions,odp_correct_handling_rate,odp_false_complete_rate,odp_unresolved_detected_rate,odp_unresolved_hidden_rate
0,rev5,ComplianceGPT,100,0.640000,0.96,0.920000,0.640000,1.00,4.010000,1.0,1.0,63,1.0,0.000000,1.000000,0.000000
1,rev5,Generative RAG baseline,100,0.220000,0.91,0.770000,0.480000,0.95,3.390000,1.0,1.0,63,0.0,0.952381,0.365079,0.634921
2,rev4,ComplianceGPT,36,0.750000,1.00,0.972222,0.750000,1.00,4.222222,1.0,1.0,19,1.0,0.000000,1.000000,0.000000
3,rev4,Generative RAG baseline,36,0.222222,1.00,0.833333,0.555556,1.00,2.722222,1.0,1.0,19,0.0,1.000000,0.315789,0.684211


,framework_version,n_compared_questions,compliancegpt_strict_win_rate,generative_false_complete_vs_compliancegpt_safe_warning_rate,generative_missed_right_control_vs_compliancegpt_rate,same_final_status_rate
0,rev5,100,0.440000,0.780000,0.05,0.170000
1,rev4,36,0.527778,0.777778,0.00,0.222222


## Metric names used in the comparison

These names are intentionally direct:

- **strict_contract_pass_rate**: final answer passed the full verifier
- **correct_control_found_rate**: answer cited at least one clause from the correct control
- **correct_clause_found_rate**: answer cited at least one correct gold clause
- **full_clause_recall_rate**: answer recovered all gold clauses for that question
- **grounded_answer_rate**: answer includes at least one evidence span
- **strict_verbatim_grounding_rate**: evidence spans pass strict verbatim grounding checks
- **false_complete_rate_on_odp_questions**: gold says the question depends on ODPs, but the system still returned `OK`
- **correct_params_required_rate_on_odp_questions**: gold says the question depends on ODPs, and the system returned `PARAMS_REQUIRED`
- **odps_surfaced_rate_on_odp_questions**: the system explicitly surfaced ODP dependence, either by returning `PARAMS_REQUIRED` or by listing required ODP ids
- **odps_suppressed_rate_on_odp_questions**: the system hid ODP dependence instead of surfacing it
- **baseline_false_complete_vs_compliancegpt_params_required_rate**: baseline said `OK` while ComplianceGPT said `PARAMS_REQUIRED` on the same question
